# 🚦 Notebook 4 — Traffic Congestion Predictor

**Project:** AI-Enabled Smart Emergency Response & Ambulance Coordination System
**Model role:** Predict the congestion level (0 = clear, 1 = standstill) for any city zone
at any point in time. The dispatch engine multiplies Google Maps' baseline ETA by this
local-pattern model so it adapts to local rhythms (school hours, market days, monsoon).

### 🎯 Performance Target
- **R² ≥ 0.95**
- **MAE ≤ 0.05** congestion units
- **Spearman ρ ≥ 0.97** (the *ordering* of zones by congestion is what matters operationally)

## 1 · Setup & Imports

In [ ]:
# !pip install -q numpy pandas scikit-learn xgboost lightgbm catboost shap matplotlib seaborn joblib

import os, json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, r2_score, mean_squared_error)
from scipy.stats import spearmanr
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("Setup complete ✔")

## 2 · Synthetic Data Generation

The generator produces realistic per-zone, per-hour congestion samples driven by:
- Time of day (rush peaks at 9 AM and 18 PM, dead at 03 AM)
- Day of week (weekend morning peaks shift later)
- Month seasonality (monsoon June–Sept worse)
- Zone density (CBD vs. suburban)
- Weather (rain inflates congestion)
- Holiday / school-day flags

In [ ]:
N = 50_000
N_ZONES = 12

def generate_traffic_dataset(n=N, n_zones=N_ZONES):
    rng = np.random.default_rng(RANDOM_STATE)

    zone_density = rng.uniform(0.2, 1.0, n_zones).round(2)   # CBD ~1.0, suburb ~0.3
    zone_lat     = 19.07 + rng.normal(0, 0.05, n_zones)
    zone_lng     = 72.87 + rng.normal(0, 0.05, n_zones)

    zone_id = rng.integers(0, n_zones, n)
    hour    = rng.integers(0, 24, n)
    dow     = rng.integers(0, 7,  n)
    month   = rng.integers(1, 13, n)

    is_weekend  = (dow >= 5).astype(int)
    is_rush     = ((dow < 5) & (((hour >= 8) & (hour <= 10)) |
                                ((hour >= 17) & (hour <= 20)))).astype(int)
    is_night    = ((hour >= 23) | (hour <= 5)).astype(int)
    is_holiday  = (rng.random(n) < 0.05).astype(int)
    is_school_day = ((dow < 5) & (~is_holiday.astype(bool))).astype(int)
    is_monsoon  = ((month >= 6) & (month <= 9)).astype(int)
    weather     = rng.choice([0,1,2,3], size=n,
                             p=[0.62, 0.22, 0.12, 0.04])
    weather_mult = np.array([1.0, 1.10, 1.30, 1.60])[weather]

    # Two daily peaks via gaussian bumps
    morn_peak = np.exp(-((hour - 9 ) ** 2) / 4.0)
    eve_peak  = np.exp(-((hour - 18) ** 2) / 4.5)
    midnight_low = 1 - np.exp(-((hour - 3) ** 2) / 8.0)

    base_diurnal = (morn_peak + eve_peak) * 0.55 + 0.05
    base_diurnal *= midnight_low
    base_diurnal *= np.where(is_weekend == 1, 0.55, 1.0)  # weekends are calmer
    base_diurnal *= zone_density[zone_id]                  # multiply by density
    base_diurnal *= weather_mult
    base_diurnal *= np.where(is_monsoon == 1, 1.10, 1.0)
    base_diurnal *= np.where(is_holiday == 1, 0.50, 1.0)

    noise = rng.normal(0, 0.025, n)
    congestion = np.clip(base_diurnal + noise, 0, 1).round(3)
    avg_speed  = np.round(np.clip(70 * (1 - 0.85 * congestion) + rng.normal(0, 2, n),
                                  5, 80), 1)

    return pd.DataFrame({
        "zone_id":      zone_id,
        "zone_density": zone_density[zone_id],
        "lat":          np.round(zone_lat[zone_id], 4),
        "lng":          np.round(zone_lng[zone_id], 4),
        "hour":         hour,
        "day_of_week":  dow,
        "month":        month,
        "is_weekend":   is_weekend,
        "is_rush_hour": is_rush,
        "is_night":     is_night,
        "is_monsoon":   is_monsoon,
        "is_holiday":   is_holiday,
        "is_school_day":is_school_day,
        "weather":      weather,
        "avg_speed_kmh":avg_speed,
        "congestion":   congestion,
    })

df = generate_traffic_dataset()
print(f"Generated {len(df):,} rows × {df.shape[1]} columns")
df.head()

## 3 · Exploratory Data Analysis

### 3.1 · Target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(df["congestion"], bins=60, color="#e67e22", edgecolor="white")
ax.set_title("Congestion distribution"); ax.set_xlabel("congestion (0–1)")
ax.axvline(df["congestion"].median(), color="red", ls="--",
           label=f"median = {df['congestion'].median():.3f}")
ax.legend(); plt.tight_layout(); plt.show()
print(df["congestion"].describe().round(3))

### 3.2 · Hourly pattern

In [ ]:
hourly = df.groupby(["hour","is_weekend"])["congestion"].mean().unstack()
hourly.columns = ["weekday","weekend"]

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(hourly.index, hourly["weekday"], marker="o", color="#c0392b", label="weekday")
ax.plot(hourly.index, hourly["weekend"], marker="o", color="#2980b9", label="weekend")
ax.fill_between([8,10],   0, 1, color="orange", alpha=0.10, label="rush windows")
ax.fill_between([17,20], 0, 1, color="orange", alpha=0.10)
ax.set_xticks(range(0,24)); ax.set_xlabel("hour of day"); ax.set_ylabel("mean congestion")
ax.set_title("Hourly congestion: weekday vs weekend"); ax.legend()
plt.tight_layout(); plt.show()

### 3.3 · Hour × Day-of-week heatmap

In [ ]:
heat = df.groupby(["day_of_week","hour"])["congestion"].mean().unstack()
heat.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
fig, ax = plt.subplots(figsize=(13, 4.5))
sns.heatmap(heat, cmap="RdYlGn_r", ax=ax, cbar_kws={"label":"congestion"})
ax.set_title("Mean congestion · hour × day-of-week"); ax.set_xlabel("hour")
plt.tight_layout(); plt.show()

### 3.4 · Per-zone variation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.boxplot(x="zone_id", y="congestion", data=df, palette="viridis", ax=ax,
            showfliers=False)
ax.set_title("Congestion distribution by zone (denser zones jam more)")
plt.tight_layout(); plt.show()

### 3.5 · Weather and seasonality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.boxplot(x="weather", y="congestion", data=df, palette="Blues", ax=axes[0])
axes[0].set_title("Weather (0=clear … 3=storm)")
sns.boxplot(x="month", y="congestion", data=df, palette="coolwarm", ax=axes[1])
axes[1].set_title("Month — monsoon (6–9) inflates")
sns.boxplot(x="is_holiday", y="congestion", data=df, palette=["#27ae60","#e74c3c"],
            ax=axes[2])
axes[2].set_title("Holiday vs working day")
plt.tight_layout(); plt.show()

### 3.6 · Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson correlation")
plt.tight_layout(); plt.show()

## 4 · Feature Engineering

We add cyclical encodings for hour and month so the model can express the
diurnal/seasonal periodicity without forcing piecewise splits.

In [ ]:
def add_features(d):
    d = d.copy()
    d["hour_sin"]  = np.sin(2*np.pi*d["hour"]/24)
    d["hour_cos"]  = np.cos(2*np.pi*d["hour"]/24)
    d["month_sin"] = np.sin(2*np.pi*d["month"]/12)
    d["month_cos"] = np.cos(2*np.pi*d["month"]/12)
    d["dow_sin"]   = np.sin(2*np.pi*d["day_of_week"]/7)
    d["dow_cos"]   = np.cos(2*np.pi*d["day_of_week"]/7)
    return d

df_fe = add_features(df)
feature_cols = [c for c in df_fe.columns if c not in ("congestion","avg_speed_kmh")]
print("Final feature count:", len(feature_cols))
print(feature_cols)

## 5 · Train/Test Split

In [ ]:
X = df_fe[feature_cols].values
y = df_fe["congestion"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)
scaler   = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train: {X_train_s.shape}  Test: {X_test_s.shape}")

## 6 · Model bake-off

In [ ]:
results = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te, do_cv=False):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    metrics = dict(
        MAE  = mean_absolute_error(y_te, preds),
        RMSE = np.sqrt(mean_squared_error(y_te, preds)),
        R2   = r2_score(y_te, preds),
        spearman = spearmanr(y_te, preds).correlation)
    if do_cv:
        cv_r2 = cross_val_score(model, X_tr, y_tr,
                                cv=KFold(3, shuffle=True, random_state=RANDOM_STATE),
                                scoring="r2", n_jobs=-1).mean()
        metrics["CV_R2"] = cv_r2
    results[name] = {"model": model, "preds": preds, **metrics}
    extra = f"  CV-R²={metrics.get('CV_R2',0):.4f}" if do_cv else ""
    print(f"{name:18s}  MAE={metrics['MAE']:.5f}  RMSE={metrics['RMSE']:.5f}  "
          f"R²={metrics['R2']:.4f}  ρ={metrics['spearman']:.4f}{extra}")
    return model

### 6.1 · Linear baseline

In [ ]:
evaluate("Ridge", Ridge(alpha=1.0, random_state=RANDOM_STATE),
         X_train_s, y_train, X_test_s, y_test)

### 6.2 · Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=400, max_depth=20,
                           min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)
evaluate("RandomForest", rf, X_train_s, y_train, X_test_s, y_test)

### 6.3 · XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=900, max_depth=7, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=2, reg_alpha=0.1, reg_lambda=1.0,
    objective="reg:squarederror", tree_method="hist",
    random_state=RANDOM_STATE, n_jobs=-1)
evaluate("XGBoost", xgb, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 6.4 · LightGBM (best for categorical-heavy data)

In [ ]:
lgb = LGBMRegressor(
    n_estimators=900, num_leaves=63, max_depth=-1,
    learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
    reg_alpha=0.1, reg_lambda=0.5,
    objective="regression", random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1)
evaluate("LightGBM", lgb, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 6.5 · CatBoost

In [ ]:
cat = CatBoostRegressor(
    iterations=900, depth=8, learning_rate=0.05,
    l2_leaf_reg=3.0, loss_function="RMSE",
    random_seed=RANDOM_STATE, verbose=False)
evaluate("CatBoost", cat, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 6.6 · Averaged ensemble

In [ ]:
preds_avg = (results["XGBoost"]["preds"]
           + results["LightGBM"]["preds"]
           + results["CatBoost"]["preds"]) / 3
metrics = dict(
    MAE      = mean_absolute_error(y_test, preds_avg),
    RMSE     = np.sqrt(mean_squared_error(y_test, preds_avg)),
    R2       = r2_score(y_test, preds_avg),
    spearman = spearmanr(y_test, preds_avg).correlation)
results["EnsembleAvg(XGB+LGB+Cat)"] = {"model":"avg","preds":preds_avg, **metrics}
print(f"EnsembleAvg          MAE={metrics['MAE']:.5f}  RMSE={metrics['RMSE']:.5f}  "
      f"R²={metrics['R2']:.4f}  ρ={metrics['spearman']:.4f}")

## 7 · Leaderboard

In [ ]:
leaderboard = pd.DataFrame([
    {"model": k, "MAE": v["MAE"], "RMSE": v["RMSE"], "R2": v["R2"],
     "spearman": v["spearman"]}
    for k, v in results.items()
]).sort_values("R2", ascending=False).reset_index(drop=True)
display(leaderboard.round(5))

best_name = leaderboard.iloc[0]["model"]
print(f"\n🏆 Best: {best_name}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
order = leaderboard.sort_values("R2")
ax.barh(order["model"], order["R2"], color="#e67e22")
ax.axvline(0.95, ls="--", color="red", label="0.95 target")
ax.set_xlim(0.5, 1.0); ax.set_xlabel("R²"); ax.legend()
ax.set_title("Traffic predictor R² leaderboard")
plt.tight_layout(); plt.show()

## 8 · Diagnostics

### 8.1 · Predicted vs Actual

In [ ]:
best_preds = results[best_name]["preds"]
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, best_preds, s=4, alpha=0.3, color="#e67e22")
ax.plot([0,1],[0,1], "r--", lw=2)
ax.set_xlabel("Actual congestion"); ax.set_ylabel("Predicted congestion")
ax.set_title(f"{best_name}: predicted vs actual")
plt.tight_layout(); plt.show()

### 8.2 · Residuals

In [ ]:
residuals = y_test - best_preds
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(best_preds, residuals, s=4, alpha=0.3, color="#8e44ad")
axes[0].axhline(0, color="red"); axes[0].set_title("Residuals vs predicted")
axes[1].hist(residuals, bins=80, color="#8e44ad", edgecolor="white")
axes[1].axvline(0, color="red"); axes[1].set_title("Residual distribution")
plt.tight_layout(); plt.show()
print(f"Residual mean={residuals.mean():.4f}  std={residuals.std():.4f}")

### 8.3 · Predicted hourly pattern (qualitative)

In [ ]:
m = results["LightGBM"]["model"]
sample_grid = []
for hour in range(24):
    for dow in [1, 5]:           # Tue (weekday) vs Sat (weekend)
        sample_grid.append({
            "zone_id": 5, "zone_density": 0.8,
            "lat": 19.07, "lng": 72.87,
            "hour": hour, "day_of_week": dow, "month": 7,
            "is_weekend": int(dow >= 5),
            "is_rush_hour": int(dow < 5 and ((8 <= hour <= 10) or (17 <= hour <= 20))),
            "is_night": int(hour >= 23 or hour <= 5),
            "is_monsoon": 1, "is_holiday": 0,
            "is_school_day": int(dow < 5),
            "weather": 0,
        })
grid = add_features(pd.DataFrame(sample_grid))
grid_X = scaler.transform(grid[feature_cols].values)
grid["pred"] = m.predict(grid_X)

fig, ax = plt.subplots(figsize=(11, 4.5))
for dow, color, lab in [(1,"#c0392b","Tue"), (5,"#2980b9","Sat")]:
    sub = grid[grid["day_of_week"] == dow]
    ax.plot(sub["hour"], sub["pred"], marker="o", color=color, label=lab)
ax.set_title("Model-predicted hourly congestion (CBD zone, July)")
ax.set_xlabel("hour"); ax.set_ylabel("predicted congestion")
ax.set_xticks(range(0,24)); ax.legend()
plt.tight_layout(); plt.show()

### 8.4 · Feature importance

In [ ]:
imp_model = results["LightGBM"]["model"]
imp = pd.Series(imp_model.feature_importances_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
imp.plot.barh(ax=ax, color="#16a085")
ax.set_title("LightGBM feature importance — what drives congestion?")
plt.tight_layout(); plt.show()

### 8.5 · SHAP global

In [ ]:
import shap
try:
    explainer = shap.TreeExplainer(imp_model)
    shap_vals = explainer.shap_values(X_test_s[:1000])
    shap.summary_plot(shap_vals, X_test_s[:1000],
                      feature_names=feature_cols, show=True)
except Exception as e:
    print("SHAP skipped:", e)

## 9 · Predict for a fresh request

In [ ]:
def predict_congestion(zone_id, hour, day_of_week, month, weather=0,
                       is_holiday=0, model_name="LightGBM"):
    m = results[model_name]["model"]
    row = {
        "zone_id": zone_id,
        "zone_density": float(df.loc[df["zone_id"]==zone_id,"zone_density"].iloc[0]),
        "lat": float(df.loc[df["zone_id"]==zone_id,"lat"].iloc[0]),
        "lng": float(df.loc[df["zone_id"]==zone_id,"lng"].iloc[0]),
        "hour": hour, "day_of_week": day_of_week, "month": month,
        "is_weekend": int(day_of_week >= 5),
        "is_rush_hour": int(day_of_week < 5 and ((8<=hour<=10) or (17<=hour<=20))),
        "is_night": int(hour >= 23 or hour <= 5),
        "is_monsoon": int(6 <= month <= 9),
        "is_holiday": is_holiday,
        "is_school_day": int(day_of_week < 5 and is_holiday == 0),
        "weather": weather,
    }
    fe = add_features(pd.DataFrame([row]))
    return float(m.predict(scaler.transform(fe[feature_cols].values))[0])

print("CBD zone 5, Tue 09:00, July, clear   →", round(predict_congestion(5, 9, 1, 7), 3))
print("CBD zone 5, Sun 09:00, July, clear   →", round(predict_congestion(5, 9, 6, 7), 3))
print("CBD zone 5, Tue 03:00, July, clear   →", round(predict_congestion(5, 3, 1, 7), 3))
print("CBD zone 5, Tue 18:00, July, storm   →", round(predict_congestion(5, 18, 1, 7, weather=3), 3))

## 10 · Persist artefacts

In [ ]:
joblib.dump(results["LightGBM"]["model"], "models/traffic_predictor.pkl")
joblib.dump(scaler,                       "models/traffic_scaler.pkl")
joblib.dump(feature_cols,                 "models/traffic_feature_cols.pkl")

final_metrics = {
    "best_model": best_name,
    "R2":        round(results[best_name]["R2"], 4),
    "MAE":       round(results[best_name]["MAE"], 5),
    "RMSE":      round(results[best_name]["RMSE"], 5),
    "spearman":  round(results[best_name]["spearman"], 4),
}
with open("reports/traffic_predictor_report.json","w") as f:
    json.dump(final_metrics, f, indent=2)
print(json.dumps(final_metrics, indent=2))
print("\n✅ Traffic predictor saved.")

## 11 · Summary

| Metric    | Target | Achieved |
|-----------|--------|----------|
| R²        | ≥ 0.95 | _see leaderboard_ |
| MAE       | ≤ 0.05 | _see leaderboard_ |
| Spearman ρ | ≥ 0.97 | _see leaderboard_ |

**Why this works:**
- Cyclical (sin/cos) encoding of hour, month, day-of-week lets the model express periodicity precisely.
- Multiplicative interactions (zone_density × diurnal × weather) suit gradient-boosted trees perfectly.
- LightGBM tends to win here thanks to native handling of low-cardinality categoricals.

→ Continue to **Notebook 5 — LSTM Hotspot Forecaster**.